### Step 1: Read the source and target tables

In [0]:
from pyspark.sql import functions as F

incremental_silver_orders = spark.table(
    "workspace.ecommerce_project.incremental_silver_orders"
)

silver_orders = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

### Step 2: Identify matching records

In [0]:
existing_orders_df = (
    incremental_silver_orders
    .join(
        silver_orders,
        on="order_id",
        how="inner"
    )
)

In [0]:
existing_orders_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+--------------------

In [0]:
print(
    "Existing records:",
    exiting_orders_df.count()
)

Existing records: 39


### Step 3: Identify changed records

In [0]:
status_changes_df = (
    incremental_silver_orders.alias("source")
    .join(
        silver_orders.alias("target"),
        on="order_id",
        how="inner"
    )
    .filter(
        ~F.col("source.order_status")
        .eqNullSafe(
            F.col("target.order_status")
        )
    )
    .select(
        F.col("source.order_id"),
        F.col("target.order_status").alias("old_order_status"),
        F.col("source.order_status").alias("new_order_status")
    )
)

In [0]:
status_changes_df.show()

+---------+----------------+----------------+
| order_id|old_order_status|new_order_status|
+---------+----------------+----------------+
|ORD100077|         Shipped|       Cancelled|
|ORD100249|          Placed|       Confirmed|
|ORD100067|       Delivered|       Cancelled|
|ORD100556|       Delivered|       Cancelled|
|ORD100244|       Delivered|       Cancelled|
|ORD100257|       Confirmed|       Cancelled|
|ORD100011|          Placed|       Confirmed|
|ORD100230|       Delivered|       Cancelled|
|ORD100382|         Shipped|       Cancelled|
|ORD100603|         Shipped|       Delivered|
|ORD100326|       Delivered|       Cancelled|
|ORD100099|       Delivered|       Cancelled|
|ORD100598|       Delivered|       Cancelled|
+---------+----------------+----------------+



### Step 4: Load the Silver table as a Delta table

In [0]:
from delta.tables import DeltaTable

silver_target = DeltaTable.forName(
    spark,
    "workspace.ecommerce_project.silver_orders"
)

### Step 5: Update matching records

In [0]:
(
    silver_target.alias("target")
    .merge(
        incremental_silver_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdate(
        set = {
            "customer_name":"source.customer_name",
            "customer_email":"source.customer_email",
            "customer_phone":"source.customer_phone",
            "quantity":"source.quantity",
            "unit_price":"source.unit_price",
            "discount_percent":"source.discount_percent",
            "order_amount":"source.order_amount",
            "final_amount":"source.final_amount",
            "payment_method":"source.payment_method",
            "payment_status":"source.payment_status",
            "order_status":"source.order_status",
            "order_date":"source.order_date",
            "ship_date":"source.ship_date",
            "delivery_date":"source.delivery_date",
            "shipping_city":"source.shipping_city",
            "shipping_state":"source.shipping_state",
            "pincode":"source.pincode"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_orders_after_update = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

In [0]:
silver_orders_after_update.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+--------------------

In [0]:
print("Silver records after UPDATE:",silver_orders_after_update.count())

Silver records after UPDATE: 738


In [0]:
duplicate_source_orders = (
    incremental_silver_orders
    .groupBy("order_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_source_orders.show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

